# B2B Export Sales Intelligence — Colab Demo

**Audience:** business, information-systems, supply-chain, and international-trade learners.

**Prerequisites:** basic Python and familiarity with customer qualification and export quotations. No API key is required.

**Learning goals:**
- score a fictional customer with transparent business rules;
- analyze an English RFQ without inventing missing facts;
- compare Gross Margin and Markup quotations;
- review decision-focused sales metrics and charts from the fictional demo database.

All companies, contacts, requirements, prices, and outcomes are fictional.

## 1. Load the project from GitHub

In Google Colab this cell clones the latest `main` branch and installs the hash-locked requirements. When run from a local checkout, first install the requirements as described in the README; this cell uses the current project directory.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

REPOSITORY_URL = "https://github.com/hql7-luo/b2b-export-sales-intelligence.git"

if Path("/content").exists():
    PROJECT_DIR = Path("/content/b2b-export-sales-intelligence")
    if PROJECT_DIR.exists():
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(PROJECT_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--require-hashes", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
else:
    candidates = (Path.cwd(), Path.cwd().parent)
    PROJECT_DIR = next((path for path in candidates if (path / "services").exists()), Path.cwd())

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from database.init_db import initialize_database
from database.seed_data import seed_demo_data
from services.dashboard import dashboard_snapshot
from services.inquiry_analyzer import analyze_inquiry_rules
from services.quotation import calculate_quotation
from services.scoring import score_customer

print(f"Project loaded from: {PROJECT_DIR}")

## 2. Score a fictional customer

The score is explainable: five visible dimensions add to 100 points. It supports qualification decisions but does not replace salesperson judgment.

In [ ]:
fictional_customer = {
    "company_name": "Fictional Meridian Retail",
    "website": "https://fictional-meridian.example",
    "email": "buyer@fictional-meridian.example",
    "phone": "+1 202 555 0147",
    "product_interest": "Custom recycled-paper notebook with foil logo",
    "import_frequency": "Quarterly",
    "estimated_purchase_volume": 24000,
    "current_stage": "Requirement Confirmed",
    "last_contact_date": "2026-07-15",
}

customer_score = score_customer(fictional_customer)
score_table = pd.DataFrame(
    {
        "Dimension": customer_score["breakdown"].keys(),
        "Score": customer_score["breakdown"].values(),
        "Reason": customer_score["dimension_reasons"].values(),
    }
)
print(f"Total score: {customer_score['total_score']}/100 | Grade: {customer_score['grade']}")
score_table

## 3. Analyze a fictional buyer inquiry

The local rule engine extracts only explicit facts, lists missing quotation inputs, flags risks, and prepares a professional English reply. It does not require an API key.

In [ ]:
fictional_inquiry = """
Hello, we are Fictional Meridian Retail in Canada. We need 2,000 custom
hardcover notebooks, A5 recycled paper, with our foil logo and individual packaging for delivery
to Toronto, Canada. Please quote a customized sample and advise T/T payment.
""".strip()

inquiry_analysis = analyze_inquiry_rules(fictional_inquiry)
analysis_table = pd.DataFrame(
    [
        {"Field": field, "Extracted value": value or "Missing"}
        for field, value in inquiry_analysis["extracted_fields"].items()
    ]
)
print(f"Completeness: {inquiry_analysis['completeness_score']}%")
print("Recommended questions:")
for question in inquiry_analysis["next_questions"]:
    print(f"- {question}")
analysis_table

## 4. Prepare the quotation

- **Gross Margin:** selling price = cost ÷ (1 − margin rate)
- **Markup:** selling price = cost × (1 + markup rate)

Gross Margin is the project default. The exchange-rate convention is `1 USD = X CNY`.

In [ ]:
quotation_inputs = {
    "product_unit_cost_cny": "18.50",
    "packaging_unit_cost_cny": "1.20",
    "quantity": 2000,
    "domestic_transport_cny": "1800",
    "export_handling_cny": "1200",
    "international_freight_cny": "9800",
    "insurance_cny": "650",
    "tariff_tax_cny": "7200",
    "platform_bank_fee_cny": "480",
    "exchange_rate_cny_per_usd": "7.20",
    "pricing_rate": "0.25",
}

gross_margin_quote = calculate_quotation(**quotation_inputs)
markup_quote = calculate_quotation(**quotation_inputs, pricing_method="markup")
quote_table = pd.DataFrame(
    [
        {
            "Incoterm": term,
            "Cost CNY": values["cost_total_cny"],
            "Gross Margin total USD": gross_margin_quote["terms"][term]["total_usd"],
            "Markup total USD": markup_quote["terms"][term]["total_usd"],
            "Gross Margin unit USD": gross_margin_quote["terms"][term]["unit_usd"],
        }
        for term, values in gross_margin_quote["terms"].items()
    ]
)
print("Estimated DDP must be verified for tariff, tax, customs, and final delivery.")
quote_table

## 5. Build a clean fictional analytics snapshot

This cell initializes a temporary SQLite database, loads the bundled fictional demo data, and calculates the same decision metrics used by the application.

In [ ]:
demo_workspace = tempfile.TemporaryDirectory()
demo_database = Path(demo_workspace.name) / "export_sales_demo.db"
initialize_database(demo_database)
demo_counts = seed_demo_data(demo_database)
analytics = dashboard_snapshot(demo_database)

metrics_table = pd.DataFrame(
    {
        "Metric": [
            "Customers",
            "Overdue follow-ups",
            "Inquiry-to-quote conversion",
            "Quotation-to-won conversion",
        ],
        "Value": [
            analytics["metrics"]["total_customers"],
            analytics["metrics"]["overdue_follow_ups"],
            f"{analytics['metrics']['inquiry_to_quote_conversion']}%",
            f"{analytics['metrics']['quotation_to_won_conversion']}%",
        ],
    }
)
print(f"Fictional demo records: {demo_counts}")
metrics_table

## 6. Visualize decision-support metrics

The charts focus on lead quality, the sales funnel, and customer geography rather than decorative activity metrics.

In [ ]:
quality = analytics["lead_quality"]
funnel = pd.DataFrame(analytics["funnel"])
countries = pd.DataFrame(analytics["countries"]).head(8)

figure = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=("Lead Quality", "Sales Funnel", "Top Customer Countries"),
)
figure.add_trace(go.Bar(x=list(quality.keys()), y=list(quality.values()), name="Lead Quality"), row=1, col=1)
figure.add_trace(go.Bar(x=funnel["stage"], y=funnel["customers"], name="Customers by stage"), row=1, col=2)
figure.add_trace(go.Bar(x=countries["label"], y=countries["value"], name="Customers by country"), row=1, col=3)
figure.update_layout(height=480, width=1200, showlegend=False, title_text="Fictional Export Sales Portfolio")
figure.update_xaxes(tickangle=-35)
figure

## 7. Validate the tutorial results

**Common pitfall:** a 25% Markup is not a 25% Gross Margin. Compare the quotation totals above before reviewing the checks below.

**Optional extension:** open the public Streamlit demo and complete the full `Analyze Inquiry → Prepare Quotation → Follow Up` workflow.

In [ ]:
assert customer_score["total_score"] == sum(customer_score["breakdown"].values())
assert inquiry_analysis["extracted_fields"]["quantity"] is not None
assert gross_margin_quote["terms"]["FOB"]["total_usd"] > markup_quote["terms"]["FOB"]["total_usd"]
assert analytics["metrics"]["total_customers"] == 20
assert analytics["demo_data_notice"] is True

validation = pd.DataFrame(
    {
        "Check": ["Customer scoring", "Inquiry analysis", "Quotation comparison", "Demo analytics", "API key"],
        "Result": ["Passed", "Passed", "Passed", "Passed", "Not required"],
    }
)
validation